# SolarPro — Calibration & Loss Analysis

A walkthrough of the SolarPro pipeline on the bundled **synthetic** demo plant:
calibrate the PVUSA model, validate it, model degradation (TDR), and analyze
performance losses.

Install first: `pip install -e ".[notebooks]"` from the repo root.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from solarpro.config import PlantConfig, ModelConfig
from solarpro import pipeline, preprocessing, metrics
from solarpro.prediction.calibration import calibrate_pvusa
from solarpro.prediction.pvusa import pvusa_power

## 1. Load configuration and data

In [ ]:
plant = PlantConfig.from_json('../data/sample/plant_config.json')
dataset = pipeline.load_sample_dataset()
print(plant)
dataset.head()

## 2. Preprocess and split

In [ ]:
operating = preprocessing.filter_operating_rows(dataset)
operating = preprocessing.clean(operating, plant.roof_height_m)
train, test = preprocessing.train_test_split_temporal(operating, 0.2)
print('train', len(train), 'test', len(test))

## 3. Calibrate PVUSA (Huber regression) and validate

In [ ]:
coeffs = calibrate_pvusa(train['gti'], train['temp_air'], train['wind_speed'], train['power'])
print('coefficients:', coeffs)

test_pred = pvusa_power(coeffs, test['gti'], test['temp_air'], test['wind_speed'])
print('test metrics:', metrics.evaluate(test['power'], test_pred).to_dict())

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(test['power'], test_pred, s=4, alpha=0.3)
lims = [0, max(test['power'].max(), test_pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Measured power (kW)'); plt.ylabel('Predicted power (kW)')
plt.title('PVUSA — predicted vs measured (test set)')
plt.show()

## 4. Full pipeline: degradation, losses and recommendations

In [ ]:
result = pipeline.run(plant, dataset, ModelConfig(), use_ollama=False)
print('loss summary:', result.loss_summary.to_dict())
print('\nrecommendations:\n', result.recommendations)

In [ ]:
daily = result.predictions[['clean_power', 'power']].resample('D').sum()
daily.columns = ['expected', 'measured']
daily.plot(figsize=(11, 4), title='Daily expected vs measured energy (kWh)')
plt.ylabel('kWh/day'); plt.show()